# v10a.24b same-kernel resume

**Use only if you interrupt v10a.24 during the current `D=<WRB|RWRB>` computation and DO NOT restart the runtime.**

This preserves the already-completed K2, N, J, C1 objects and all global Haar caches, installs the cross-polarization 1x1 fix, and resumes at D. It does not rerun the earlier sections.


In [ ]:
# HODGE v10a.24b — SAME-KERNEL RESUME AFTER INTERRUPTING v10a.24 D
# DO NOT restart the Colab runtime.
#
# Preconditions:
#   - v10a.24 is currently/was running in this same kernel.
#   - sections through K2, N, J, C1 completed.
#   - you interrupted during D=<WRB|RWRB>.
#
# This patch:
#   1) replaces ONLY the defective _v10a17_endpoint_vector;
#   2) keeps all existing global Haar caches alive;
#   3) resumes at D only, so K2/N/J/C1 are NOT recomputed;
#   4) then continues the blind shape + independent finite-cluster oracle.
#
# Cross-polarization one-face matches now fall through to the ordinary Haar
# contractor. The analytic -13/896 D11 shortcut remains same-polarization only.

_required = [
    "W2Ls","R2Ls","K2_cols","K2_full","N_cols","J_cols","C1_cols",
    "_v23_endpoint_cols","_v23_cols_to_full","_v23_herm_gate",
    "_v10a10_canon_block_pair","_v10a11_oneface_axial_character",
    "_v10a13_haar_factor","_qcache","anchor_faces","faces","verts","gates"
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError(
        "SAME-KERNEL resume prerequisites missing: " + ", ".join(_missing) +
        ". Do NOT use this after restarting the runtime; use the full v10a.24b notebook instead."
    )

print("="*132)
print("v10a.24b SAME-KERNEL HOTFIX/RESUME")
print("="*132)
print("K2/N/J/C1 already present : YES")
print("global Haar caches        : PRESERVED")
print("restarting at             : D=<WRB|RWRB>")
print()

def _v10a17_endpoint_vector(left_labeled, right_labeled, left_root, right_root, label=''):
    """Return the full translation-resolved bilinear vector.

    For every lattice translation dv of the left anchored half-history this
    computes

        < T_dv left | right >

    after exact H0/signature filtering, whole-block orbit collapse, individual
    Haar-pair topology collapse, and factorized SU(3) Haar contraction.

    The returned dictionary is keyed by dv in the cubic translation group.
    Haar values are cached globally across all polarization pairs, so the first
    column pays most of the tensor cost and the remaining 8 columns largely
    reuse the same canonical local topologies.
    """
    heartbeat=float(os.environ.get('V10A17_HEARTBEAT','15'))
    pair_tol=float(os.environ.get('V10A17_PAIR_TOL','2e-14'))

    RB=_v17_phys_index(right_labeled)
    Litems=[]
    for SL,st in left_labeled.items():
        for key,lv in _v10a3_physical_blocks(st).items():
            Litems.append((SL,key,lv))
    Litems.sort(key=lambda z:len(z[2]))

    # PHASE 1: H0/signature/translation scan, retaining displacement multiplicity.
    tasks={}
    matches=raw_pair_upper=skipped11=0
    skipped_dv=Counter()

    # v10a.24b correctness hotfix:
    # The analytic D11=-13/896 replacement is a SAME-POLARIZATION local
    # one-face result.  Cross-polarization 1x1 H0-signature matches are legal
    # and must fall through to the ordinary canonical-pair/Haar contractor.
    left_pol=tuple(faces[int(left_root)][1:])
    right_pol=tuple(faces[int(right_root)][1:])
    same_pol=(left_pol==right_pol)
    cross11_fallback=0
    cross11_dv=Counter()

    t0=time.time(); last=t0
    for ii,(SL,(cs,Esg),lv) in enumerate(Litems):
        for dv in verts:
            tcs=_v10a2_sig_canon(_v10a3_translate_sig(cs,dv))
            candidates=RB.get((tcs,Esg),())
            if not candidates:
                continue
            tv={_v10a3_translate_state(st,dv):float(c) for st,c in lv.items()}
            for SR,rv in candidates:
                matches += 1
                raw_pair_upper += len(tv)*len(rv)
                if len(SL)==1 and len(SR)==1:
                    if same_pol:
                        skipped11 += 1
                        skipped_dv[tuple(dv)] += 1
                        continue
                    # Cross-polarization 1x1 matches are NOT represented by the
                    # analytic same-polarization D11 block.  Keep them in the
                    # ordinary exact topology collapse / Haar path.
                    cross11_fallback += 1
                    cross11_dv[tuple(dv)] += 1
                key,A,B=_v10a10_canon_block_pair(tv,rv)
                rec=tasks.get(key)
                if rec is None:
                    rec=[A,B,Counter(),len(A)*len(B)]
                    tasks[key]=rec
                rec[2][tuple(dv)] += 1
        now=time.time()
        if V10A7_PROGRESS and (now-last>=heartbeat or ii+1==len(Litems)):
            rate=(ii+1)/max(now-t0,1e-9)
            eta=(len(Litems)-ii-1)/rate if rate else 0.0
            print(f'      {label} scan {ii+1:,}/{len(Litems):,}; matches={matches:,}; '
                  f'unique-blocks={len(tasks):,}; raw-pairs<={raw_pair_upper:,}; '
                  f'RAM={_v10a8_mem_gb():.2f} GiB; elapsed={now-t0:.1f}s ETA~{eta:.1f}s',flush=True)
            last=now

    out=defaultdict(float)

    # Firewall the shortcut itself.  For cross-polarization endpoint pairs the
    # skipped counter MUST remain zero; any 1x1 matches must have gone through
    # the general contractor above.
    gate(f'{label} analytic one-face shortcut confined to same polarization',
         same_pol or skipped11==0,
         f'left_pol={left_pol}, right_pol={right_pol}, '
         f'analytic_skips={skipped11}, crosspol_fallback={cross11_fallback}')

    if not same_pol and cross11_fallback:
        print(f'      {label} cross-polarization 1x1 fallback: '
              f'matches={cross11_fallback}; displacements={dict(cross11_dv)}',
              flush=True)

    # The analytically solved one-face C- block is strictly local and is used
    # ONLY for same-polarization endpoint pairs.  Center flux then forces the
    # translated bra plaquette to coincide with the ket plaquette.
    if skipped11:
        if not same_pol:
            raise RuntimeError(
                f'{label}: INTERNAL ERROR — cross-polarization one-face match '
                f'was incorrectly routed to analytic D11'
            )
        lp=faces[int(left_root)]
        rp=faces[int(right_root)]
        dv0=tuple((int(rp[0][i])-int(lp[0][i]))%L for i in range(3))
        if set(skipped_dv)-{dv0}:
            raise RuntimeError(f'{label}: one-face matches escaped the local displacement: {dict(skipped_dv)}')
        d11=float(_v10a11_oneface_axial_character()['D'])
        out[dv0] += d11
        gate(f'{label} one-face direct block=-13/896',
             abs(d11+13/896)<V10A7_TOL,d11)

    # PHASE 2: collapse individual canonical Haar pair topologies while keeping a
    # sparse displacement-weight ledger for each topology.
    pairw={}
    pair_occ=0
    tp=time.time(); last=tp
    taskrecs=list(tasks.values())
    for ti,(A,B,dv_mult,cost) in enumerate(taskrecs):
        ga=defaultdict(list); gb=defaultdict(list)
        for st,c in A: ga[_v9_flux_key_state(st)].append((st,c))
        for st,c in B: gb[_v9_flux_key_state(st)].append((st,c))
        for fk,la in ga.items():
            lb=gb.get(fk)
            if not lb:
                continue
            for aa,ca in la:
                for bb,cb in lb:
                    x,y=_joint_canon_states(aa,bb)
                    kx=(x.occ,x.part); ky=(y.occ,y.part)
                    if ky<kx:
                        x,y=y,x
                    pk=(x,y)
                    dw=pairw.get(pk)
                    if dw is None:
                        dw=defaultdict(float)
                        pairw[pk]=dw
                    base=float(ca)*float(cb)
                    for dv,mult in dv_mult.items():
                        dw[dv] += base*float(mult)
                    pair_occ += sum(dv_mult.values())
        now=time.time()
        if V10A7_PROGRESS and (now-last>=heartbeat or ti+1==len(taskrecs)):
            rate=(ti+1)/max(now-tp,1e-9)
            eta=(len(taskrecs)-ti-1)/rate if rate else 0.0
            print(f'      {label} collapse {ti+1:,}/{len(taskrecs):,}; '
                  f'pair-occ={pair_occ:,}; unique={len(pairw):,}; '
                  f'RAM={_v10a8_mem_gb():.2f} GiB; elapsed={now-tp:.1f}s ETA~{eta:.1f}s',flush=True)
            last=now

    # Drop topology/displacement weights that cancel before Haar.
    compact={}
    sparse_weights=0
    for k,d in pairw.items():
        z={dv:w for dv,w in d.items() if abs(w)>pair_tol}
        if z:
            compact[k]=z
            sparse_weights += len(z)
    pairw=compact
    print(f'      {label} PAIR-TOPOLOGY COLLAPSE: occurrences={pair_occ:,} -> '
          f'unique_nonzero={len(pairw):,}; sparse displacement weights={sparse_weights:,}; '
          f'RAM={_v10a8_mem_gb():.2f} GiB',flush=True)

    # Cold dense/factorized equivalence on every endpoint-pattern signature.
    reps={}
    for (a,b) in pairw:
        pats=tuple(sorted(_v10_endpoint_patterns(a,b)))
        reps.setdefault(pats,(a,b))
    ferr=[]
    for _pats,(a,b) in reps.items():
        dense=float(_qcache(a,b))
        fact=float(_v10a13_haar_factor(a,b))
        ferr.append(abs(dense-fact))
    gate(f'{label} factorized Haar matches dense endpoint signatures',
         max(ferr,default=0.0)<5e-12,
         f'maxerr={max(ferr,default=0.0):.3e}, signatures={len(reps)}')

    # PHASE 3: one Haar value per canonical topology; LRU is intentionally kept
    # live across polarization pairs.
    items=list(pairw.items())
    items.sort(key=lambda kv:_v10a13_pair_score(kv[0][0],kv[0][1]),reverse=True)
    te=time.time(); last=te
    c0=_v10a13_haar_factor.cache_info()
    for ii,((a,b),weights) in enumerate(items,1):
        h=_v10a17_factor_haar_cached(a,b)
        if h:
            for dv,w in weights.items():
                out[dv] += h*w
        now=time.time()
        if V10A7_PROGRESS and (now-last>=heartbeat or ii%2000==0 or ii==len(items)):
            ci=_v10a13_haar_factor.cache_info()
            rate=ii/max(now-te,1e-9); eta=(len(items)-ii)/rate if rate else 0.0
            print(f'      {label} Haar {ii:,}/{len(items):,}; '
                  f'cache hits/misses={ci.hits:,}/{ci.misses:,}; '
                  f'elapsed={now-te:.1f}s ETA~{eta:.1f}s; RAM={_v10a8_mem_gb():.2f} GiB',flush=True)
            last=now
    c1=_v10a13_haar_factor.cache_info()

    out={dv:float(x) for dv,x in out.items() if abs(x)>2e-13}
    total=float(sum(out.values()))
    print(f'      {label} COMPLETE gamma-sum={total:+.15g}; nonzero translations={len(out)}; '
          f'new Haar misses={c1.misses-c0.misses:,}; reused hits={c1.hits-c0.hits:,}; '
          f'total={time.time()-t0:.1f}s',flush=True)
    return out,dict(
        left_blocks=len(Litems),matched_blocks=matches,raw_pair_upper=raw_pair_upper,
        unique_block_pairs=len(tasks),pair_occurrences=pair_occ,
        unique_haar_pairs=len(pairw),sparse_displacement_weights=sparse_weights,
        oneface_matches=skipped11,haar_new_misses=c1.misses-c0.misses,
        haar_reused_hits=c1.hits-c0.hits,total_seconds=time.time()-t0
    )


def _v10a17_build_K2(anchor_w1,anchor_r1):
    """Small exact-H0 second-order endpoint kernel; no fourth-order work here."""
    r1blocks=[_v10a3_physical_blocks(x) for x in anchor_r1]
    pol_index={pol:i for i,pol in enumerate(T1_POLS)}
    K2=np.zeros((P,3),dtype=np.float64)
    t0=time.time(); matches=0
    for f,(v,a,b) in enumerate(faces):
        bi=pol_index[(a,b)]
        tw1=_v10a3_translate_h0_state(anchor_w1[bi],v)
        for ai in range(3):
            K2[f,ai],m=_v10a3_h0_state_inner(tw1,anchor_r1[ai],qhaar,r1blocks[ai])
            matches += m
        if V10A7_PROGRESS and (f+1)%max(1,P//5)==0:
            print(f'      v10a17 K2 {f+1:,}/{P:,}; H0 matches={matches:,}; elapsed={time.time()-t0:.1f}s',flush=True)
    return K2


def _v10a8_gamma_support_inner(left_labeled,right_labeled,label='',skip_support_sizes=()):
    """Gamma bilinear directly from support-resolved halves.

    This avoids the 432-term aggregate H0 blocks created when 171 local supports
    are merged before the Haar contraction.  Exact H0 signatures and lattice
    translations still provide the first orthogonality filter; whole local block
    pairs are then jointly canonicalized and memoized.
    """
    RB=_v17_phys_index(right_labeled)
    Litems=[]
    for SL,st in left_labeled.items():
        if len(SL) in set(skip_support_sizes): continue
        for key,lv in _v10a3_physical_blocks(st).items():
            Litems.append((SL,key,lv))
    # Small blocks first: immediate progress and cache warm-up before determinant blocks.
    Litems.sort(key=lambda z: len(z[2]))
    ans=0.0; sig_tests=matched=0; raw_pair_upper=0
    block_cache={}; cache_hits=cache_misses=haar_pairs=0
    t0=time.time(); last=t0
    for ii,(SL,(cs,Esg),lv) in enumerate(Litems):
        for dv in verts:
            sig_tests += 1
            tcs=_v10a2_sig_canon(_v10a3_translate_sig(cs,dv))
            candidates=RB.get((tcs,Esg),())
            if not candidates: continue
            tv={_v10a3_translate_state(st,dv):float(c) for st,c in lv.items()}
            for SR,rv in candidates:
                if len(SR) in set(skip_support_sizes): continue
                matched += 1; raw_pair_upper += len(tv)*len(rv)
                x,hit,npairs=_v10a8_block_key_and_inner(tv,rv,_qcache,block_cache)
                ans += x
                if hit: cache_hits += 1
                else: cache_misses += 1; haar_pairs += npairs
        now=time.time()
        if V10A7_PROGRESS and (now-last >= V10A8_HEARTBEAT or ii+1==len(Litems)):
            rate=(ii+1)/max(now-t0,1e-9); eta=(len(Litems)-ii-1)/rate if rate>0 else float('inf')
            print(f"      {label} heartbeat local-blocks={ii+1:,}/{len(Litems):,} size={len(lv):,} "
                  f"matches={matched:,} cache={len(block_cache):,} hits={cache_hits:,} misses={cache_misses:,} "
                  f"haar-pairs={haar_pairs:,} RAM={_v10a8_mem_gb():.2f} GiB elapsed={now-t0:.1f}s ETA~{eta/60:.1f}m",flush=True)
            last=now
    return float(ans),dict(left_blocks=len(Litems),signature_tests=sig_tests,matched_h0_blocks=matched,
        raw_pair_upper_bound=raw_pair_upper,block_cache_entries=len(block_cache),block_cache_hits=cache_hits,
        block_cache_misses=cache_misses,unique_haar_pairs=haar_pairs,elapsed=time.time()-t0)

def _v10a8_gamma_inner(left,right,haar,label=''):
    """Fast exact Gamma-point bilinear for v10a.8.

    Mathematical value is identical to _v10a4_gamma_inner.  Optimization is
    representational only: exact H0/translation matching first, whole-block
    joint canonicalization second, memoized local Haar contraction third.
    """
    LB=_v10a3_physical_blocks(left)
    RB=_v10a3_physical_blocks(right)
    ans=0.0; sig_tests=matched=0; raw_pair_upper=0
    block_cache={}
    cache_hits=cache_misses=haar_pairs=0
    t0=time.time(); last=t0
    items=list(LB.items())
    _ls=sorted((len(v) for v in LB.values()), reverse=True); _rs=sorted((len(v) for v in RB.values()), reverse=True)
    print(f'      {label} census: LB={len(LB):,} RB={len(RB):,} max-terms L/R={(_ls[0] if _ls else 0):,}/{(_rs[0] if _rs else 0):,} topL={_ls[:8]} topR={_rs[:8]}', flush=True)
    # Exact right lookup is already O(1); the 125 signature tests are cheap and
    # are retained because they are a very strong orthogonality prefilter.
    for ii,((cs,Esg),lv) in enumerate(items):
        for dv in verts:
            sig_tests += 1
            tcs=_v10a2_sig_canon(_v10a3_translate_sig(cs,dv))
            rv=RB.get((tcs,Esg))
            if rv is None:
                continue
            matched += 1
            tv={_v10a3_translate_state(st,dv):float(c) for st,c in lv.items()}
            raw_pair_upper += len(tv)*len(rv)
            x,hit,npairs=_v10a8_block_key_and_inner(tv,rv,_qcache,block_cache)
            ans += x
            if hit: cache_hits += 1
            else:
                cache_misses += 1
                haar_pairs += npairs
        now=time.time()
        if V10A7_PROGRESS and (now-last >= V10A8_HEARTBEAT or ii+1==len(items)):
            rate=(ii+1)/max(now-t0,1e-9)
            eta=(len(items)-ii-1)/rate if rate>0 else float('inf')
            print(f"      {label} heartbeat blocks={ii+1:,}/{len(items):,} "
                  f"matches={matched:,} block-cache={len(block_cache):,} "
                  f"hits={cache_hits:,} misses={cache_misses:,} haar-pairs={haar_pairs:,} "
                  f"RAM={_v10a8_mem_gb():.2f} GiB elapsed={now-t0:.1f}s ETA~{eta/60:.1f}m")
            last=now
    return float(ans),dict(left_blocks=len(LB),right_blocks=len(RB),
        signature_tests=sig_tests,matched_h0_blocks=matched,
        raw_pair_upper_bound=raw_pair_upper,block_cache_entries=len(block_cache),
        block_cache_hits=cache_hits,block_cache_misses=cache_misses,
        unique_haar_pairs=haar_pairs,elapsed=time.time()-t0)

# =============================================================================
# HODGE v10a.7 — MARKED LINKED O(u^4) SCALAR LEDGER
# =============================================================================
# PURPOSE
# -------
# Close the remaining SU(3) Gamma-point fourth-order scalar problem without a
# global Q2 Gram build.  This layer combines the already-certified ingredients:
#
#   * full-complement four-moment Feshbach formula
#         e4 = D - 2 e1 C - e2 N + e1^2 J,
#   * exact source-sector identity Q1 W Q1 = 0, hence sigma3_A=C_A=0,
#   * extended SU(3) Haar endpoint tensors through (6,0)/(0,6),
#   * cold adjacent determinant certificate,
#   * protected O(u^4) Hodge mobility values as independent firewalls.
#
# New ingredient here: SUPPORT-RESOLVED marked histories.
# Instead of rerunning every finite cluster separately, each magnetic history is
# tagged by the exact plaquette support it uses through the two-insertion half
# histories.  The labels certify the complete rooted local corpus; the final
# Gamma Haar bilinear is then evaluated after H0-block merging for tractability.
#
# Vacuum O(u^4) is treated independently.  A hard Z3/cubical-cycle gate proves
# that a nonzero connected four-insertion vacuum support uses at most two
# distinct plaquettes.  We therefore evaluate the one-face and the two adjacent
# pair vacuum clusters exactly, subtract their proper subclusters, and count
# their embeddings attached to one marked plaquette.
#
# NO historical m4 value is supplied anywhere in the construction.
# The final m4_rest is printed only after all hard gates pass.
# =============================================================================

from fractions import Fraction as _Q17
from collections import defaultdict as _dd17, Counter as _Counter17, deque as _deque17
import itertools as _it17
import gc as _gc17

V10A7_GATE_START = len(gates)
V10A7_PROGRESS = int(os.environ.get('V10A7_PROGRESS','1'))
V10A7_TOL = float(os.environ.get('V10A7_TOL','3e-9'))
V10A7_SHAPE_TOL = float(os.environ.get('V10A7_SHAPE_TOL','5e-7'))
V10A7_RAT_DEN = int(os.environ.get('V10A7_RAT_DEN','1000000000'))
V10A7_DO_SHAPE = os.environ.get('V10A7_DO_SHAPE','1') != '0'
V10A7_SUPPORT_POLS = tuple(int(x) for x in os.environ.get('V10A7_SUPPORT_POLS','0,1,2').split(',') if x.strip())
if not V10A7_SUPPORT_POLS or any(x not in (0,1,2) for x in V10A7_SUPPORT_POLS):
    raise ValueError('V10A7_SUPPORT_POLS must be a nonempty subset of 0,1,2')
V10A7_UNBLIND = os.environ.get('V10A7_UNBLIND','1') != '0'
V10A7_RECHECK_Q1 = os.environ.get('V10A7_RECHECK_Q1','0') != '0'

if L != 5:
    raise RuntimeError(f'v10a.7 requires GLUE_L=5 for the cold shape firewall; got L={L}')

print('='*136)
print('HODGE v10a.23 — PREREQUISITE HODGE/HAAR CORPUS + DUAL COLD ORACLES')
print('='*136)
print(f'backend                         : {DEVICE}')
print(f'lattice                         : L={L}')
print('global K=2 P/Q Gram build       : REMOVED')
print('global physical Q2 basis        : NOT REQUIRED')
print('Gamma source fold identity      : C_A = sigma3_A = 0 (separate exact certificate)')
print('history organization            : exact rooted half-support census + merged H0/Haar Gamma bilinear')
print('vacuum linked subtraction       : one-face + adjacent-pair clusters after Z3 cycle completeness gate')
print('historical m4 target            : NOT LOADED')
print('support-resolved polarizations  :',V10A7_SUPPORT_POLS,'(cubic representatives; set 0,1,2 for full redundancy)')
print('recheck already-exact Q1 moments:',V10A7_RECHECK_Q1)

# -----------------------------------------------------------------------------
# 0. Geometry helpers
# -----------------------------------------------------------------------------

_V17_FID = {(tuple(v),int(a),int(b)):i for i,(v,a,b) in enumerate(faces)}
_V17_NEIGH = [frozenset(face_support_faces[f]) for f in range(P)]  # includes self


def _v17_translate_face(f,dv):
    v,a,b=faces[int(f)]
    vv=((v[0]+int(dv[0]))%L,(v[1]+int(dv[1]))%L,(v[2]+int(dv[2]))%L)
    return _V17_FID[(vv,int(a),int(b))]


def _v17_translate_support(S,dv):
    return frozenset(_v17_translate_face(f,dv) for f in S)


def _v17_faces_share_link(a,b):
    if int(a)==int(b): return True
    return int(b) in _V17_NEIGH[int(a)]


def _v17_connected(S):
    S=set(map(int,S))
    if not S: return True
    seen={next(iter(S))}; q=_deque17(seen)
    while q:
        a=q.popleft()
        for b in S-seen:
            if _v17_faces_share_link(a,b):
                seen.add(b); q.append(b)
    return len(seen)==len(S)


def _v17_face_normal(f):
    _,a,b=faces[int(f)]
    return ({0,1,2}-{int(a),int(b)}).pop()


def _v17_pair_class(a,b):
    if not _v17_faces_share_link(a,b) or int(a)==int(b):
        raise ValueError('pair class requires two distinct shared-link plaquettes')
    return 'coplanar' if _v17_face_normal(a)==_v17_face_normal(b) else 'perpendicular'


def _v17_rational(x,maxden=V10A7_RAT_DEN):
    return _Q17(float(x)).limit_denominator(int(maxden))

# -----------------------------------------------------------------------------
# 1. Vacuum support-completeness theorem gate at four insertions
# -----------------------------------------------------------------------------
# A vacuum Haar endpoint must have zero linkwise Z3 boundary.  On the cubic
# complex there is no nonzero local 2-cycle on <=4 faces (the elementary cube
# is the first one, at six faces).  We cold-check the <=4 statement on every
# rooted connected support and every nonzero Z3 coefficient assignment.


def _v17_rooted_connected_subsets(root,maxsize=4):
    root=int(root)
    levels={1:{frozenset((root,))}}
    allsets=set(levels[1])
    for k in range(2,int(maxsize)+1):
        nxt=set()
        for S in levels[k-1]:
            frontier=set()
            for f in S: frontier.update(_V17_NEIGH[f])
            for g in frontier-set(S):
                T=frozenset(set(S)|{int(g)})
                if len(T)==k and _v17_connected(T): nxt.add(T)
        levels[k]=nxt; allsets.update(nxt)
    return levels,allsets

_cycle_root=next(f for f,(v,a,b) in enumerate(faces) if v==(0,0,0) and (a,b)==(0,1))
_v17_levels,_v17_subsets=_v17_rooted_connected_subsets(_cycle_root,4)
_nonzero_small_cycles=[]
for S in _v17_subsets:
    fs=tuple(sorted(S))
    if len(fs)>4: continue
    A=np.asarray(B2[:,fs],dtype=np.int16)
    for coeff in _it17.product((1,2),repeat=len(fs)):
        z=(A@np.asarray(coeff,dtype=np.int16))%3
        if np.all(z==0):
            _nonzero_small_cycles.append((fs,coeff)); break

gate('v10a.7 no nonzero rooted Z3 cubical 2-cycle exists on <=4 faces',
     len(_nonzero_small_cycles)==0,
     f"counts={dict((k,len(v)) for k,v in _v17_levels.items())}, bad={len(_nonzero_small_cycles)}")

# If the four-insertion vacuum chain itself is Z3-trivial, enumerate its signed
# multiplicities.  Every occupied face must have net coefficient 0 mod 3.
_max_support=0; _good_mult=[]
for k in range(1,5):
    for labels in _it17.product(range(k),repeat=4):
        if set(labels)!=set(range(k)): continue
        for signs in _it17.product((-1,+1),repeat=4):
            net=[0]*k
            for a,s in zip(labels,signs): net[a]+=s
            if all(x%3==0 for x in net):
                _max_support=max(_max_support,k); _good_mult.append((labels,signs))
gate('v10a.7 four-insertion Z3-trivial vacuum multiplicities use <=2 distinct faces',
     _max_support<=2,f'max distinct support={_max_support}, valid patterns={len(_good_mult)}')

# -----------------------------------------------------------------------------
# 2. Install the already-certified extended SU(3) Haar contractor
# -----------------------------------------------------------------------------

qhaar,qsupported,qcert,_qcache=_v10a2_install_q2_haar(globals())
gate('v10a.7 k=3 Weingarten inverse',qcert['wg3_inverse_error']<5e-13,f"err={qcert['wg3_inverse_error']:.3e}")
gate('v10a.7 pure-six singlet rank=5',qcert['rank60']==5,f"rank={qcert['rank60']}")
gate('v10a.7 pure-six projector idempotent',qcert['T60_idempotence_error']<5e-13,f"err={qcert['T60_idempotence_error']:.3e}")

_fsmodel=_v10a4_fs_model()

# -----------------------------------------------------------------------------
# 3. Generic restricted-W and vacuum reduced resolvent
# -----------------------------------------------------------------------------


def _v17_apply_W_faces(state,allowed_faces):
    """W=-M restricted to an explicit finite set of magnetic plaquettes."""
    out={}; actions=channels=0
    F=tuple(sorted(map(int,allowed_faces)))
    for sig,v in state.items():
        for f in F:
            for o in (-1,+1):
                actions+=1
                for pv,osig in _v10a3_project_action_dyn(_fsmodel,v,sig,f,o):
                    channels+=1; _v10a3_sig_vec_add(out,osig,pv,-1.0)
    return _v10a3_compress_state(out),dict(actions=actions,channels=channels)


_V17_VAC_ST=LXState((),())
_V17_VAC_VEC={_V17_VAC_ST:1.0}
_V17_VAC={():dict(_V17_VAC_VEC)}


def _v17_vac_R(state,label='Rv'):
    """Q(0-H0)^-1Q with exact vacuum projection in the E=0 block."""
    blocks=_v10a3_physical_blocks(state)
    resmax=0.0; ng=0
    for (cs,Esg),v0 in blocks.items():
        if Esg!=0: continue
        ng+=1; v=dict(v0)
        ov=_v10a3_vec_inner(_V17_VAC_VEC,v,qhaar)
        if abs(ov)>V10A3_COEFF_TOL:
            for st,c in _V17_VAC_VEC.items(): v[st]=v.get(st,0.0)-ov*c
        v=_v10a3_prune(v)
        resmax=max(resmax,_v10a3_vec_norm(v,qhaar))
    if resmax>V10A3_NORM_TOL:
        raise RuntimeError(f'{label}: unresolved E=0 vacuum residual norm2={resmax:.3e}')
    out={}
    for sig,v in state.items():
        Esg=_v10a2_sig_E(sig)
        if Esg==0: continue
        den=-float(Esg)
        out[sig]={st:float(c)/den for st,c in v.items()}
    return _v10a3_compress_state(out),dict(E0_groups=ng,E0_residual_norm2_max=resmax)


def _v17_inner(a,b):
    return _v10a3_h0_state_inner(a,b,qhaar)[0]


def _v17_vac_cluster(C):
    C=frozenset(map(int,C))
    w1,_=_v17_apply_W_faces(_V17_VAC,C)
    r1,s1=_v17_vac_R(w1,'vac R1')
    w2,_=_v17_apply_W_faces(r1,C)
    r2,s2=_v17_vac_R(w2,'vac R2')
    e1=_v17_inner(_V17_VAC,w1)
    e2=_v17_inner(w1,r1)
    sig3=_v17_inner(r1,w2)
    Nn=_v17_inner(r1,r1)
    Dd=_v17_inner(w2,r2)
    e3=sig3 # e1=0
    e4=Dd-e2*Nn
    return dict(C=C,e1=e1,e2=e2,sigma3=sig3,e3=e3,N=Nn,D=Dd,e4=e4,
                pole=max(s1['E0_residual_norm2_max'],s2['E0_residual_norm2_max']))

# One-face vacuum is the exact compact character laboratory, now through the
# same Wilson/H0/Haar history engine used below.
_vac_seed=_cycle_root
_v1=_v17_vac_cluster({_vac_seed})
print('\n[3] COLD ONE-FACE VACUUM CLUSTER')
for k in ('e1','e2','sigma3','e3','N','D','e4'):
    print(f'  {k:7s} = {_v1[k]:+.15g}  rational~ {_v17_rational(_v1[k])}')
gate('v10a.7 one-face vacuum e1=0',abs(_v1['e1'])<V10A7_TOL,_v1['e1'])
gate('v10a.7 one-face vacuum e2=-3/4',abs(_v1['e2']+3/4)<V10A7_TOL,_v1['e2'])
gate('v10a.7 one-face vacuum e3=-9/32',abs(_v1['e3']+9/32)<V10A7_TOL,_v1['e3'])
gate('v10a.7 one-face vacuum N=9/32',abs(_v1['N']-9/32)<V10A7_TOL,_v1['N'])
gate('v10a.7 one-face vacuum D=-309/1280',abs(_v1['D']+309/1280)<V10A7_TOL,_v1['D'])
gate('v10a.7 one-face vacuum e4=-39/1280',abs(_v1['e4']+39/1280)<V10A7_TOL,_v1['e4'])

# Two adjacent vacuum cluster representatives.
_pair_rep={}
for q in sorted(_V17_NEIGH[_vac_seed]-{_vac_seed}):
    cls=_v17_pair_class(_vac_seed,q)
    _pair_rep.setdefault(cls,q)
if set(_pair_rep)!={'coplanar','perpendicular'}:
    raise RuntimeError(f'failed to find both vacuum adjacent pair classes: {_pair_rep}')

_vpair={}
print('\n[4] COLD TWO-FACE VACUUM CLUSTERS')
for cls,q in sorted(_pair_rep.items()):
    z=_v17_vac_cluster({_vac_seed,q})
    linked=z['e4']-2.0*_v1['e4']
    linked_e2=z['e2']-2.0*_v1['e2']
    linked_e3=z['e3']-2.0*_v1['e3']
    _vpair[cls]=dict(full=z,linked_e4=linked,linked_e2=linked_e2,linked_e3=linked_e3)
    print(f'  {cls:13s}: e4(C)={z["e4"]:+.15g} ~ {_v17_rational(z["e4"])}; '
          f'omega4={linked:+.15g} ~ {_v17_rational(linked)}')
    gate(f'v10a.7 vacuum {cls} pair has no linked O(u^2)',abs(linked_e2)<V10A7_TOL,linked_e2)
    gate(f'v10a.7 vacuum {cls} pair has no linked O(u^3)',abs(linked_e3)<V10A7_TOL,linked_e3)
    gate(f'v10a.7 vacuum {cls} pair full e4=-54321/837760',abs(z['e4']-float(_Q17(-54321,837760)))<V10A7_TOL,z['e4'])
    gate(f'v10a.7 vacuum {cls} pair omega4=-327/83776',abs(linked-float(_Q17(-327,83776)))<V10A7_TOL,linked)

gate('v10a.7 coplanar/perpendicular vacuum pair linked weights agree',abs(_vpair['coplanar']['linked_e4']-_vpair['perpendicular']['linked_e4'])<V10A7_TOL,_vpair['coplanar']['linked_e4']-_vpair['perpendicular']['linked_e4'])

# Exact disconnected-spectator check: a two-face vacuum cluster with no shared
# link must factorize, so its irreducible pair weight is identically zero.
_far=next(f for f in range(P) if f!=_vac_seed and f not in _V17_NEIGH[_vac_seed])
_vfar=_v17_vac_cluster({_vac_seed,_far})
_vfar_omega=_vfar['e4']-2.0*_v1['e4']
print(f'  disconnected pair spectator omega4={_vfar_omega:+.3e}')
gate('v10a.7 disconnected two-face vacuum spectator has zero linked O(u^4)',abs(_vfar_omega)<V10A7_TOL,_vfar_omega)

# Count vacuum linked-cluster embeddings attached to one marked plaquette.
_mark=_vac_seed
_single_emb=sorted(_V17_NEIGH[_mark])
_pairs=set()
for a in range(P):
    for b in _V17_NEIGH[a]:
        b=int(b)
        if b<=a: continue
        if not _v17_faces_share_link(a,b): continue
        if _v17_connected({_mark,a,b}): _pairs.add(frozenset((a,b)))
_pair_counts=_Counter17(_v17_pair_class(*tuple(S)) for S in _pairs)
print('\n[5] VACUUM EMBEDDINGS ATTACHED TO ONE MARKED PLAQUETTE')
print('  one-face embeddings:',len(_single_emb))
print('  adjacent-pair embeddings:',dict(_pair_counts),'total=',len(_pairs))
gate('v10a.7 one-face vacuum embedding count is 13',len(_single_emb)==13,len(_single_emb))

V4_LINKED_MARKED = len(_single_emb)*_v1['e4']
for cls,n in _pair_counts.items(): V4_LINKED_MARKED += int(n)*_vpair[cls]['linked_e4']
print(f'  linked vacuum O4 subtraction around mark = {V4_LINKED_MARKED:+.15g}  rational~ {_v17_rational(V4_LINKED_MARKED)}')
_V4_EX=_Q17(-1474623,1675520)
gate('v10a.7 linked vacuum O4 subtraction=-1474623/1675520',abs(V4_LINKED_MARKED-float(_V4_EX))<V10A7_TOL,V4_LINKED_MARKED)

# -----------------------------------------------------------------------------
# 4. Support-resolved connected axial recursion
# -----------------------------------------------------------------------------


def _v17_add_state(dst,src,scale=1.0):
    for sig,v in src.items(): _v10a3_sig_vec_add(dst,sig,v,scale)


def _v17_aggregate(LD):
    out={}
    for st in LD.values(): _v17_add_state(out,st,1.0)
    return _v10a3_compress_state(out)


def _v17_apply_W_labeled(LD,label='W'):
    out={}; actions=channels=0
    items=list(LD.items()); t0=time.time()
    for ii,(S,state) in enumerate(items):
        for sig,v in state.items():
            cand=_v10a3_candidate_faces_vec(v)
            for f in cand:
                for o in (-1,+1):
                    actions+=1
                    for pv,osig in _v10a3_project_action_dyn(_fsmodel,v,sig,int(f),int(o)):
                        channels+=1
                        T=frozenset(set(S)|{int(f)})
                        d=out.setdefault(T,{})
                        _v10a3_sig_vec_add(d,osig,pv,-1.0)
        if V10A7_PROGRESS and len(items)>20 and (ii+1)%max(1,len(items)//4)==0:
            print(f'      {label} supports {ii+1}/{len(items)}; out supports={len(out)}; elapsed={time.time()-t0:.1f}s')
    return {S:_v10a3_compress_state(st) for S,st in out.items() if _v10a3_compress_state(st)},dict(actions=actions,channels=channels)


def _v17_R_labeled(LD,label='R'):
    out={}; mx=0.0; ng=0
    for S,st in LD.items():
        rr,ss=_v10a3_reduced_resolvent(st,qhaar,label)
        mx=max(mx,float(ss['E0_residual_norm2_max'])); ng+=int(ss['E0_groups'])
        if rr: out[S]=rr
    return out,dict(E0_groups=ng,E0_residual_norm2_max=mx)


def _v17_phys_index(LD):
    idx=_dd17(list)
    for S,st in LD.items():
        for key,v in _v10a3_physical_blocks(st).items(): idx[key].append((S,v))
    return idx


def _v17_gamma_ledger(left,right,label='Gamma'):
    """Return exact-support ledger for <left_Gamma|right_Gamma>.

    Right source remains anchored.  Each translated left support is unioned with
    the right support before accumulation, so every contribution is assigned to
    its minimal marked plaquette support.
    """
    RB=_v17_phys_index(right)
    ledger=_dd17(float); tests=matches=haar_terms=0; t0=time.time()
    Litems=[]
    for SL,st in left.items():
        for key,lv in _v10a3_physical_blocks(st).items(): Litems.append((SL,key,lv))
    for ii,(SL,(cs,Esg),lv) in enumerate(Litems):
        for dv in verts:
            tests+=1
            tcs=_v10a2_sig_canon(_v10a3_translate_sig(cs,dv))
            candidates=RB.get((tcs,Esg),())
            if not candidates: continue
            matches+=len(candidates)
            tv={_v10a3_translate_state(st,dv):float(c) for st,c in lv.items()}
            tSL=_v17_translate_support(SL,dv)
            for SR,rv in candidates:
                x=_v10a3_vec_inner(tv,rv,qhaar)
                if abs(x)>1e-16:
                    ledger[frozenset(set(tSL)|set(SR))]+=float(x)
                haar_terms+=len(tv)*len(rv)
        if V10A7_PROGRESS and len(Litems)>200 and (ii+1)%max(1,len(Litems)//5)==0:
            print(f'      {label} blocks {ii+1}/{len(Litems)}; H0 matches={matches}; supports={len(ledger)}; elapsed={time.time()-t0:.1f}s')
    ledger={S:x for S,x in ledger.items() if abs(x)>2e-13}
    return ledger,dict(left_blocks=len(Litems),signature_tests=tests,matched_h0_blocks=matches,haar_upper=haar_terms)


def _v17_sum(L): return float(sum(L.values()))


def _v17_add_ledger(dst,src,scale=1.0):
    for S,x in src.items(): dst[S]+=float(scale)*float(x)


def _v17_union_convolution(A,B):
    out=_dd17(float)
    for SA,a in A.items():
        for SB,b in B.items(): out[frozenset(set(SA)|set(SB))]+=float(a)*float(b)
    return {S:x for S,x in out.items() if abs(x)>2e-13}


def _v17_size_summary(ledger):
    c=_Counter17(); w=_dd17(float)
    for S,x in ledger.items(): c[len(S)]+=1; w[len(S)]+=float(x)
    return dict(sorted(c.items())),dict(sorted(w.items()))

# Anchors are exactly the three origin T1 faces used by the endpoint/Bloch code.
anchor_faces=[next(f for f,(v,a,b) in enumerate(faces) if v==(0,0,0) and (a,b)==pol) for pol in T1_POLS]
print('\n[6] SUPPORT-RESOLVED AXIAL HALF-HISTORIES')
print('  anchor faces:',[(f,faces[f]) for f in anchor_faces])

S0s={}; W1Ls={}; R1Ls={}; W2Ls={}; R2Ls={}; R12Ls={}
W1s={};R1s={};W2s={};R2s={};R12s={}
resmax=0.0
for a in V10A7_SUPPORT_POLS:
    p0=anchor_faces[a]
    print(f'  polarization {a} {T1_POLS[a]}')
    S0={frozenset((p0,)):_v10a3_face_state(p0)}; S0s[a]=S0
    W1L,sw1=_v17_apply_W_labeled(S0,f'W1 pol{a}')
    R1L,sr1=_v17_R_labeled(W1L,f'R1 pol{a}')
    W2L,sw2=_v17_apply_W_labeled(R1L,f'W2 pol{a}')
    R2L,sr2=_v17_R_labeled(W2L,f'R2 pol{a}')
    R12L,sr12=_v17_R_labeled(R1L,f'R(R1) pol{a}')
    resmax=max(resmax,sr1['E0_residual_norm2_max'],sr2['E0_residual_norm2_max'],sr12['E0_residual_norm2_max'])
    W1Ls[a]=W1L;R1Ls[a]=R1L;W2Ls[a]=W2L;R2Ls[a]=R2L;R12Ls[a]=R12L
    W1s[a]=_v17_aggregate(W1L);R1s[a]=_v17_aggregate(R1L);W2s[a]=_v17_aggregate(W2L);R2s[a]=_v17_aggregate(R2L);R12s[a]=_v17_aggregate(R12L)
    print('    support counts W1/R1/W2/R2 =',len(W1L),len(R1L),len(W2L),len(R2L))
    print('    aggregate state stats W1/R1/W2/R2 =',_v10a3_state_stats(W1s[a]),_v10a3_state_stats(R1s[a]),_v10a3_state_stats(W2s[a]),_v10a3_state_stats(R2s[a]))

gate('v10a.7 all support-resolved reduced resolvents are free of E0 poles',resmax<V10A7_TOL,f'max residual norm2={resmax:.3e}')


# =============================================================================
# HODGE v10a.23 — DEGENERATE-FOLD FULL-T1 K4 OPERATOR
# =============================================================================
# This is the first layer in this solver that is allowed to form a fourth-order
# *operator* on the complete one-flux model space.  It does not use either
# disputed fourth-order scalar, nor the historical fourth-order C-shape target.
#
# Definitions (R = Q(E0-H0)^-1 Q):
#   K2 = B^T R B
#   N  = B^T R^2 B
#   J  = B^T R^3 B
#   D  = B^T R W R W R B
#   C1 = B^T R^2 W R B
# If PVP = a I on the full one-flux P space, the Hermitian canonical
# (des-Cloizeaux / SW) fourth-order block is
#
#   H4 = D - a(C1+C1^T) - 1/2 {K2,N} + a^2 J.
#
# The notebook first derives/regression-tests this operator identity on exact
# rational finite Hamiltonians with dim(P)=2.  It then proves/falsifies PVP=aP
# in the physical Wilson-network model before applying the formula.
# =============================================================================

import random as _v23_random
import sympy as _v23_sp
import hashlib as _v23_hashlib

V23_GATE_START=len(gates)
V23_TOL=float(os.environ.get('V10A23_TOL','3e-8'))
V23_RECORD_TOL=float(os.environ.get('V10A23_RECORD_TOL','2e-8'))
V23_HEART=float(os.environ.get('V10A23_HEARTBEAT','15'))

print('\n'+'='*148)
print('HODGE v10a.24b — DEGENERATE-FOLD FULL-T1 FOURTH-ORDER OPERATOR')
print('='*148)
print('disputed q/rest targets          : NOT LOADED')
print('historical C-shape target        : NOT LOADED')
print('PVP=aP                           : COMPUTED, NOT ASSUMED')
print('folded formula                   : EXACT dim(P)>1 SW REGRESSION FIRST')
print('full T1 polarizations            : 3')

# -----------------------------------------------------------------------------
# 1. Exact degenerate Schrieffer-Wolff regression of the operator formula
# -----------------------------------------------------------------------------

def _v23_smul(A,B):
    order=len(A)-1; n=A[0].rows
    out=[_v23_sp.zeros(n) for _ in range(order+1)]
    for k in range(order+1):
        z=_v23_sp.zeros(n)
        for i in range(k+1): z += A[i]*B[k-i]
        out[k]=z
    return out


def _v23_scomm(A,B):
    AB=_v23_smul(A,B); BA=_v23_smul(B,A)
    return [AB[i]-BA[i] for i in range(len(A))]


def _v23_bch(H,S,order):
    out=[x.copy() for x in H]; X=[x.copy() for x in H]; fac=1
    for k in range(1,order+1):
        X=_v23_scomm(X,S); fac*=k
        out=[out[i]+X[i]/_v23_sp.Integer(fac) for i in range(order+1)]
    return out


def _v23_sw_exact(H0,V,p,order=4):
    n=H0.rows
    H=[_v23_sp.zeros(n) for _ in range(order+1)]; H[0]=H0; H[1]=V
    S=[_v23_sp.zeros(n) for _ in range(order+1)]
    E0=H0[0,0]
    for r in range(1,order+1):
        base=_v23_bch(H,S,order)[r]
        Sr=_v23_sp.zeros(n)
        for q in range(p,n):
            den=H0[q,q]-E0
            for j in range(p):
                Sr[q,j]=-_v23_sp.cancel(base[q,j]/den)
                Sr[j,q]=-Sr[q,j]
        S[r]=Sr
    Hf=_v23_bch(H,S,order)
    return [_v23_sp.simplify(Hf[r][:p,:p]) for r in range(order+1)]


def _v23_fold_formula_exact(H0,V,p):
    E0=H0[0,0]; a=V[0,0]
    B=V[p:,:p]; W=V[p:,p:]
    R=_v23_sp.diag(*[_v23_sp.cancel(1/(E0-H0[i,i])) for i in range(p,H0.rows)])
    K2=_v23_sp.simplify(B.T*R*B)
    N =_v23_sp.simplify(B.T*(R**2)*B)
    J =_v23_sp.simplify(B.T*(R**3)*B)
    C1=_v23_sp.simplify(B.T*(R**2)*W*R*B)
    D =_v23_sp.simplify(B.T*R*W*R*W*R*B)
    K3=_v23_sp.simplify(B.T*R*W*R*B-a*N)
    H4=_v23_sp.simplify(D-a*(C1+C1.T)-_v23_sp.Rational(1,2)*(K2*N+N*K2)+a*a*J)
    return K2,K3,H4


def _v23_degenerate_fold_regression():
    rows=[]
    for seed in (2301,2302,2303,2304):
        rr=_v23_random.Random(seed); p=2; n=5
        H0=_v23_sp.diag(0,0,2,3,5)
        a=_v23_sp.Integer(rr.randint(-2,2))
        V=_v23_sp.zeros(n); V[:p,:p]=a*_v23_sp.eye(p)
        for i in range(p,n):
            for j in range(p):
                x=_v23_sp.Integer(rr.randint(-2,2)); V[i,j]=V[j,i]=x
        for i in range(p,n):
            for j in range(i,n):
                x=_v23_sp.Integer(rr.randint(-2,2)); V[i,j]=V[j,i]=x
        sw=_v23_sw_exact(H0,V,p,4)
        K2,K3,H4=_v23_fold_formula_exact(H0,V,p)
        B=V[p:,:p]; R=_v23_sp.diag(*[_v23_sp.cancel(1/(H0[0,0]-H0[i,i])) for i in range(p,n)])
        N=B.T*(R**2)*B
        noncomm=(_v23_sp.simplify(K2*N-N*K2)!=_v23_sp.zeros(p))
        e2=_v23_sp.simplify(sw[2]-K2)
        e3=_v23_sp.simplify(sw[3]-K3)
        e4=_v23_sp.simplify(sw[4]-H4)
        ok=(e2==_v23_sp.zeros(p) and e3==_v23_sp.zeros(p) and e4==_v23_sp.zeros(p))
        rows.append((seed,a,ok,noncomm))
    return rows

print('\n[10] EXACT DEGENERATE FOLD REGRESSION')
_v23_reg=_v23_degenerate_fold_regression()
gate('v10a.23 exact dim(P)=2 SW regressions prove folded operator formula',
     all(x[2] for x in _v23_reg),_v23_reg)
gate('v10a.23 toy regressions genuinely test noncommuting K2 and N',all(x[3] for x in _v23_reg),_v23_reg)

# -----------------------------------------------------------------------------
# 2. Physical PVP=aP gate — computed from the actual one-W trace-network action
# -----------------------------------------------------------------------------
print('\n[11] PHYSICAL PVP MODEL-SPACE GATE')
PVP_anchor=np.column_stack([_v10a3_project_P0(W1s[a],qhaar) for a in range(3)])
PVP_round=np.rint(PVP_anchor)
pvp_lift_err=float(np.max(np.abs(PVP_anchor-PVP_round)))
PVP_expected=np.zeros_like(PVP_round)
for a,f in enumerate(anchor_faces): PVP_expected[int(f),a]=1.0
pvp_struct_err=float(np.max(np.abs(PVP_round-PVP_expected)))
gate('v10a.23 physical PVP coefficients lift unambiguously to integers',pvp_lift_err<2e-10,pvp_lift_err)
gate('v10a.23 physical PVP is exactly +I under translation covariance',pvp_struct_err==0.0,
     f'lift_err={pvp_lift_err:.3e}, structural_err={pvp_struct_err:.1f}')
A_PVP=1.0

# -----------------------------------------------------------------------------
# 3. General translation-resolved pair-collapsed bilinear (no D-only shortcut)
# -----------------------------------------------------------------------------

def _v23_endpoint_general(left_labeled,right_labeled,left_root,right_root,label=''):
    heartbeat=V23_HEART
    pair_tol=float(os.environ.get('V10A23_PAIR_TOL','2e-14'))
    RB=_v17_phys_index(right_labeled)
    Litems=[]
    for SL,st in left_labeled.items():
        for key,lv in _v10a3_physical_blocks(st).items(): Litems.append((SL,key,lv))
    Litems.sort(key=lambda z:len(z[2]))

    tasks={}; matches=raw_pair_upper=0
    t0=time.time(); last=t0
    for ii,(SL,(cs,Esg),lv) in enumerate(Litems):
        for dv in verts:
            tcs=_v10a2_sig_canon(_v10a3_translate_sig(cs,dv))
            candidates=RB.get((tcs,Esg),())
            if not candidates: continue
            tv={_v10a3_translate_state(st,dv):float(c) for st,c in lv.items()}
            for SR,rv in candidates:
                matches+=1; raw_pair_upper+=len(tv)*len(rv)
                key,A,B=_v10a10_canon_block_pair(tv,rv)
                rec=tasks.get(key)
                if rec is None:
                    rec=[A,B,Counter()]; tasks[key]=rec
                rec[2][tuple(dv)]+=1
        now=time.time()
        if V10A7_PROGRESS and (now-last>=heartbeat or ii+1==len(Litems)):
            rate=(ii+1)/max(now-t0,1e-9); eta=(len(Litems)-ii-1)/rate if rate else 0.0
            print(f'      {label} scan {ii+1:,}/{len(Litems):,}; matches={matches:,}; '
                  f'blocks={len(tasks):,}; raw-pairs<={raw_pair_upper:,}; elapsed={now-t0:.1f}s ETA~{eta:.1f}s',flush=True)
            last=now

    pairw={}; pair_occ=0
    recs=list(tasks.values()); tp=time.time(); last=tp
    for ti,(A,B,dv_mult) in enumerate(recs):
        ga=defaultdict(list); gb=defaultdict(list)
        for st,c in A: ga[_v9_flux_key_state(st)].append((st,c))
        for st,c in B: gb[_v9_flux_key_state(st)].append((st,c))
        multsum=sum(dv_mult.values())
        for fk,la in ga.items():
            lb=gb.get(fk)
            if not lb: continue
            for aa,ca in la:
                for bb,cb in lb:
                    x,y=_joint_canon_states(aa,bb)
                    if (y.occ,y.part)<(x.occ,x.part): x,y=y,x
                    pk=(x,y); dw=pairw.get(pk)
                    if dw is None: dw=defaultdict(float); pairw[pk]=dw
                    base=float(ca)*float(cb)
                    for dv,mult in dv_mult.items(): dw[dv]+=base*float(mult)
                    pair_occ+=multsum
        now=time.time()
        if V10A7_PROGRESS and (now-last>=heartbeat or ti+1==len(recs)):
            print(f'      {label} collapse {ti+1:,}/{len(recs):,}; pair-occ={pair_occ:,}; unique={len(pairw):,}; elapsed={now-tp:.1f}s',flush=True)
            last=now

    pairw={k:{dv:w for dv,w in d.items() if abs(w)>pair_tol} for k,d in pairw.items()}
    pairw={k:d for k,d in pairw.items() if d}
    reps={}
    for a,b in pairw:
        pats=tuple(sorted(_v10_endpoint_patterns(a,b))); reps.setdefault(pats,(a,b))
    ferr=[]
    for a,b in reps.values(): ferr.append(abs(float(_qcache(a,b))-float(_v10a13_haar_factor(a,b))))
    gate(f'{label} factorized Haar matches dense endpoint signatures',max(ferr,default=0.0)<5e-12,
         f'maxerr={max(ferr,default=0.0):.3e}, signatures={len(reps)}')

    out=defaultdict(float); items=list(pairw.items())
    items.sort(key=lambda kv:_v10a13_pair_score(kv[0][0],kv[0][1]),reverse=True)
    te=time.time(); last=te
    for ii,((a,b),weights) in enumerate(items,1):
        h=_v10a17_factor_haar_cached(a,b)
        if h:
            for dv,w in weights.items(): out[dv]+=h*w
        now=time.time()
        if V10A7_PROGRESS and (now-last>=heartbeat or ii%2000==0 or ii==len(items)):
            rate=ii/max(now-te,1e-9); eta=(len(items)-ii)/rate if rate else 0.0
            print(f'      {label} Haar {ii:,}/{len(items):,}; elapsed={now-te:.1f}s ETA~{eta:.1f}s',flush=True); last=now
    out={dv:float(x) for dv,x in out.items() if abs(x)>2e-13}
    print(f'      {label} COMPLETE gamma-sum={sum(out.values()):+.15g}; translations={len(out)}; total={time.time()-t0:.1f}s',flush=True)
    return out,dict(left_blocks=len(Litems),matched_blocks=matches,raw_pair_upper=raw_pair_upper,
                    unique_block_pairs=len(tasks),pair_occurrences=pair_occ,unique_haar_pairs=len(pairw),
                    total_seconds=time.time()-t0,
                    analytic_oneface_matches=skipped11,
                    crosspol_oneface_fallback_matches=cross11_fallback,
                    crosspol_oneface_fallback_displacements=dict(cross11_dv))



D_cols,D_stats=_v23_endpoint_cols(W2Ls,R2Ls,'D=<WRB|RWRB>',d_special=True)

N_full,_=_v23_herm_gate('N',_v23_cols_to_full(N_cols))
J_full,_=_v23_herm_gate('J',_v23_cols_to_full(J_cols))
D_full,_=_v23_herm_gate('D',_v23_cols_to_full(D_cols))
C1_full=_v23_cols_to_full(C1_cols)
L_full=C1_full+C1_full.T

H4_ax_full=(D_full-A_PVP*L_full-0.5*(K2_full@N_full+N_full@K2_full)+(A_PVP*A_PVP)*J_full)
H4_ax_full,_=_v23_herm_gate('folded axial H4',H4_ax_full,5*V23_TOL)
H4_ax_cols=_v23_full_to_cols(H4_ax_full)

print('\n[13] FOLDED AXIAL BLOCH/HODGE SHAPE — STILL BLIND')
shape23=_v10a3_extract_shape(H4_ax_cols)
for k in ('rest_direct','A','B','C_direct','D','alpha','fifth_residual_max','hermiticity_error','gamma_spread'):
    print(f'  {k:22s} = {shape23[k]:+.15g}')

gate('v10a.23 folded axial Gamma block is cubic/scalar',shape23['gamma_spread']<5e-7,shape23['gamma_spread'])
gate('v10a.23 folded axial real-space kernel is Hermitian',shape23['hermiticity_error']<5e-7,shape23['hermiticity_error'])
gate('v10a.23 protected A3^shp=5/48',abs(shape23['A']-5/48)<5e-6,shape23['A'])
gate('v10a.23 protected B3^shp=0',abs(shape23['B'])<5e-6,shape23['B'])
gate('v10a.23 protected D3^shp=0',abs(shape23['D'])<5e-6,shape23['D'])
gate('v10a.23 protected alpha3=5/12',abs(shape23['alpha']-5/12)<2e-5,shape23['alpha'])
gate('v10a.23 folded full-shape residual closes',shape23['fifth_residual_max']<1e-5,shape23['fifth_residual_max'])

# Gamma scalar check against the independently exact one-polarization direct/fold
# moments.  This is a regression only; it is NOT called a physical mass.
D_gamma=float(np.trace(_v10a3_bloch_from_anchor(D_cols,(0,0,0))).real/3.0)
N_gamma=float(np.trace(_v10a3_bloch_from_anchor(N_cols,(0,0,0))).real/3.0)
J_gamma=float(np.trace(_v10a3_bloch_from_anchor(J_cols,(0,0,0))).real/3.0)
C_gamma=float(np.trace(_v10a3_bloch_from_anchor(C1_cols,(0,0,0))).real/3.0)
H4_gamma_formula=D_gamma-2*A_PVP*C_gamma-float(_Q17(-5945,612))*N_gamma+A_PVP*A_PVP*J_gamma
print('  Gamma moment audit:')
print('    D =',D_gamma,' C =',C_gamma,' N =',N_gamma,' J =',J_gamma)
print('    scalar folded formula =',H4_gamma_formula)
print('    matrix H4 Gamma       =',shape23['rest_direct'])
gate('v10a.23 matrix fold reduces to scalar Feshbach formula at Gamma',
     abs(H4_gamma_formula-shape23['rest_direct'])<2e-6,
     abs(H4_gamma_formula-shape23['rest_direct']))

# Persist blind outputs for the independent linked oracle below.
V23_AXIAL_H4_COLS=H4_ax_cols
V23_AXIAL_SHAPE=shape23
V23_AXIAL_RECORDS=int(np.count_nonzero(np.abs(H4_ax_cols)>V23_RECORD_TOL))
print('  blind folded axial nonzero anchored records =',V23_AXIAL_RECORDS)

# =============================================================================
# HODGE v10a.24b — INDEPENDENT ROOTED FINITE-CLUSTER LINKED-GAP ORACLE
# =============================================================================
# This second leg does NOT use any of the operator-moment ledgers above to obtain
# the physical rest coefficient.  The W1/W2 histories are used only as a SUPPORT
# census.  Each raw finite-cluster coefficient is recomputed by diagonalizing
# H_C(u)=H0+uV_C on that restricted cluster, constructing the Hermitian
# des-Cloizeaux one-particle block, subtracting an independently diagonalized
# vacuum level, and only then performing rooted incidence subtraction.
#
# The disputed fourth-order values remain absent until all lower-order and
# numerical-stability gates pass.
# =============================================================================

import itertools as _v23c_it
import math as _v23c_math
import time as _v23c_time
from collections import defaultdict as _v23c_dd, Counter as _v23c_Counter
from fractions import Fraction as _V23CF
from numpy.polynomial import Polynomial as _V23Poly
from scipy.linalg import eigh as _v23c_eigh

V23C_POL=int(os.environ.get('V10A23_CLUSTER_POL','2'))
V23C_ROOT=int(anchor_faces[V23C_POL])
V23C_GRAM_TOL=float(os.environ.get('V10A23_CLUSTER_GRAM_TOL','2e-10'))
V23C_HERM_TOL=float(os.environ.get('V10A23_CLUSTER_HERM_TOL','2e-8'))
V23C_FIT_UMAX=float(os.environ.get('V10A23_CLUSTER_FIT_UMAX','0.055'))
V23C_FIT_DEG=int(os.environ.get('V10A23_CLUSTER_FIT_DEG','6'))
V23C_FIT_N=int(os.environ.get('V10A23_CLUSTER_FIT_N','13'))
V23C_FIT_STAB_TOL=float(os.environ.get('V10A23_CLUSTER_FIT_STAB_TOL','5e-3'))
V23C_SYM_CHECKS=int(os.environ.get('V10A23_CLUSTER_SYM_CHECKS','4'))
V23C_PROGRESS=int(os.environ.get('V10A23_CLUSTER_PROGRESS','1'))

if V23C_FIT_N < V23C_FIT_DEG+3 or V23C_FIT_N%2==0:
    raise ValueError('V10A23_CLUSTER_FIT_N must be odd and at least FIT_DEG+3')

print('\n'+'='*148)
print('HODGE v10a.23 — INDEPENDENT FINITE-CLUSTER LINKED-GAP ORACLE')
print('='*148)
print('root/polarization                :',V23C_ROOT,faces[V23C_ROOT],V23C_POL)
print('raw cluster coefficients         : RECOMPUTED FROM RESTRICTED HAMILTONIANS')
print('operator H4 moment ledger        : NOT USED FOR REST COEFFICIENT')
print('disputed fourth-order values     : STILL NOT LOADED')
print('Krylov closure                   : P + Q1 + Q2')
print('fit                              : symmetric polynomial, umax=',V23C_FIT_UMAX,'deg=',V23C_FIT_DEG,'N=',V23C_FIT_N)

# ------------------------------- state algebra -------------------------------
def _v23c_state_copy(a):
    return {sig:{st:float(c) for st,c in v.items()} for sig,v in a.items()}

def _v23c_state_add(dst,src,scale=1.0):
    s=float(scale)
    for sig,v in src.items():
        d=dst.setdefault(sig,{})
        for st,c in v.items(): d[st]=d.get(st,0.0)+s*float(c)
    return _v10a3_compress_state(dst)

def _v23c_state_scaled(a,scale):
    z={}; return _v23c_state_add(z,a,scale)

def _v23c_inner(a,b): return float(_v10a3_h0_state_inner(a,b,qhaar)[0])

def _v23c_norm2(a):
    x=_v23c_inner(a,a)
    if x < -5e-9: raise RuntimeError(f'negative Haar norm2 in cluster oracle: {x:.3e}')
    return max(0.0,x)

def _v23c_split_h0(state):
    out={}
    for sig,v in state.items():
        key=(_v10a2_sig_canon(sig),_v10a2_sig_E(sig))
        d=out.setdefault(key,{})
        d[sig]={st:float(c) for st,c in v.items()}
    ans={}
    for k,v in out.items():
        z=_v10a3_compress_state(v)
        if z: ans[k]=z
    return ans

def _v23c_applyW(state,C): return _v17_apply_W_faces(state,C)[0]

# -------------------------- finite-cluster Krylov basis ----------------------
def _v23c_build_basis(C,vacuum=False):
    C=frozenset(map(int,C))
    if not C: raise ValueError('empty finite cluster')
    pfaces=tuple(sorted(C)) if not vacuum else ()
    initial=[_v23c_state_copy(_V17_VAC)] if vacuum else [_v23c_state_copy(_v10a3_face_state(f)) for f in pfaces]
    basis=[]; bykey=_v23c_dd(list); p_indices=[]

    def add_raw(raw,key,layer,name):
        v=_v23c_state_copy(raw)
        for _ in range(2):
            for i in bykey.get(key,()):
                ov=_v23c_inner(basis[i]['state'],v)
                if abs(ov)>1e-13: _v23c_state_add(v,basis[i]['state'],-ov)
        n2=_v23c_norm2(v)
        if n2<=V23C_GRAM_TOL: return None
        v=_v23c_state_scaled(v,1.0/_v23c_math.sqrt(n2))
        i=len(basis); basis.append({'state':v,'key':key,'layer':int(layer),'name':str(name)}); bykey[key].append(i)
        return i

    for j,s in enumerate(initial):
        blocks=_v23c_split_h0(s)
        if len(blocks)!=1: raise RuntimeError(f'P seed split into {len(blocks)} H0 blocks')
        key,blk=next(iter(blocks.items()))
        i=add_raw(blk,key,0,'vac' if vacuum else f'Pface{pfaces[j]}')
        if i is None: raise RuntimeError('lost independent P seed')
        p_indices.append(i)
    if p_indices!=list(range(len(p_indices))): raise RuntimeError('P ordering invariant failed')
    nP=len(p_indices)

    layer1=[]
    for j in range(nP):
        w=_v23c_applyW(basis[j]['state'],C)
        for key,blk in _v23c_split_h0(w).items():
            i=add_raw(blk,key,1,f'W(P{j})')
            if i is not None: layer1.append(i)
    layer2=[]
    for jj,j in enumerate(layer1):
        w=_v23c_applyW(basis[j]['state'],C)
        for key,blk in _v23c_split_h0(w).items():
            i=add_raw(blk,key,2,f'W(Q1{jj})')
            if i is not None: layer2.append(i)

    Eref=_V23CF(0,1) if vacuum else _V23CF(8,3)
    resonant=[i for i,b in enumerate(basis[nP:],start=nP) if b['key'][1]==Eref]
    if resonant: raise RuntimeError(f'finite cluster retained {len(resonant)} non-P resonances at E0={Eref}')

    nb=len(basis); H0=np.asarray([float(b['key'][1]) for b in basis],float); W=np.zeros((nb,nb),float)
    for j,b in enumerate(basis):
        wb=_v23c_applyW(b['state'],C); blocks=_v23c_split_h0(wb)
        for key,blk in blocks.items():
            for i in bykey.get(key,()): W[i,j]=_v23c_inner(basis[i]['state'],blk)
    herm=float(np.max(np.abs(W-W.T))) if nb else 0.0
    if herm>V23C_HERM_TOL: raise RuntimeError(f'finite-cluster W not Hermitian: {herm:.3e}, |C|={len(C)}, dim={nb}')
    W=0.5*(W+W.T)

    PG=np.zeros((nP,nP),float)
    for i in range(nP):
        for j in range(i,nP): PG[i,j]=PG[j,i]=_v23c_inner(basis[i]['state'],basis[j]['state'])
    perr=float(np.max(np.abs(PG-np.eye(nP))))
    if perr>2e-8: raise RuntimeError(f'finite-cluster P basis lost orthonormality: {perr:.3e}')
    return {'C':C,'vacuum':bool(vacuum),'pfaces':pfaces,'nP':nP,'basis':basis,'H0':H0,'W':W,'dim':nb,
            'layer1':len(layer1),'layer2':len(layer2),'herm':herm,'pgram_err':perr}

# -------------------------- des-Cloizeaux projection -------------------------
def _v23c_one_particle_heff(model,u):
    nP=int(model['nP']); H=np.diag(model['H0'])+float(u)*model['W']
    ew,U=_v23c_eigh(H,check_finite=False,overwrite_a=False)
    pweight=np.sum(U[:nP,:]**2,axis=0); sel=np.argsort(pweight)[-nP:]
    A=U[:nP,sel]; S=0.5*(A.T@A+(A.T@A).T); se,SV=np.linalg.eigh(S)
    if float(se[0])<0.20: raise RuntimeError(f'des-Cloizeaux P projection nearly singular: min={se[0]:.3e}, u={u}')
    Sinv=SV@np.diag(1.0/np.sqrt(se))@SV.T; Phi=A@Sinv
    orth=float(np.max(np.abs(Phi.T@Phi-np.eye(nP))))
    if orth>2e-8: raise RuntimeError(f'des-Cloizeaux projected vectors not orthonormal: {orth:.3e}')
    Heff=Phi@np.diag(ew[sel])@Phi.T; Heff=0.5*(Heff+Heff.T)
    return Heff,float(np.sum(pweight[sel])),float(se[0])

def _v23c_vac_energy(model,u):
    H=np.diag(model['H0'])+float(u)*model['W']; ew,U=_v23c_eigh(H,check_finite=False,overwrite_a=False)
    j=int(np.argmax(U[0,:]**2)); return float(ew[j]),float(U[0,j]**2)

def _v23c_cluster_gap_value(one,vac,u):
    Heff,pw,smin=_v23c_one_particle_heff(one,u); Ev,vw=_v23c_vac_energy(vac,u)
    root_i=one['pfaces'].index(V23C_ROOT); _,ra,rb=faces[V23C_ROOT]
    same=[i for i,f in enumerate(one['pfaces']) if tuple(faces[int(f)][1:])==(ra,rb)]
    return float(np.sum(Heff[root_i,same])-Ev),dict(pweight=pw,pSmin=smin,vweight=vw)

def _v23c_fit_cluster(C):
    one=_v23c_build_basis(C,False); vac=_v23c_build_basis(C,True)
    us=np.linspace(-V23C_FIT_UMAX,V23C_FIT_UMAX,V23C_FIT_N); ys=[]; qmin=1.0; vmin=1.0
    for u in us:
        y,st=_v23c_cluster_gap_value(one,vac,float(u)); ys.append(y); qmin=min(qmin,st['pSmin']); vmin=min(vmin,st['vweight'])
    ys=np.asarray(ys,float); p=_V23Poly.fit(us,ys,deg=V23C_FIT_DEG).convert()
    c=np.zeros(5,float); c[:min(5,len(p.coef))]=p.coef[:5]
    mask=np.abs(us)<=0.76*V23C_FIT_UMAX+1e-15; deg2=min(V23C_FIT_DEG,int(np.count_nonzero(mask))-2)
    p2=_V23Poly.fit(us[mask],ys[mask],deg=deg2).convert(); c2=np.zeros(5,float); c2[:min(5,len(p2.coef))]=p2.coef[:5]
    z0,_=_v23c_cluster_gap_value(one,vac,0.0)
    if abs(z0-8/3)>2e-9: raise RuntimeError(f'finite-cluster u=0 gap incorrect: {z0}')
    return {'coef':c,'coef_inner':c2,'fit_stability':float(abs(c[4]-c2[4])),'one_dim':one['dim'],'vac_dim':vac['dim'],
            'q1':one['layer1'],'q2':one['layer2'],'one_herm':one['herm'],'vac_herm':vac['herm'],'pSmin':qmin,'vweight':vmin}

# --------------------------- support candidate census ------------------------
def _v24c_candidate_supports(leftLD,rightLD,left_root,right_root,max_size=7):
    """Return concrete supports AND the translated left P endpoint.

    This is purely a support census: coefficients/Haar weights are ignored.
    Keeping the endpoint is essential for folded K2*N support composition,
    because the second two-W segment must be translated to the actual
    intermediate P plaquette rather than naively unioned at the original root.
    """
    RB=_v17_phys_index(rightLD)
    out=set()
    by_endpoint=_v23c_dd(set)
    tests=matches=0
    Litems=[]
    for SL,st in leftLD.items():
        for key,lv in _v10a3_physical_blocks(st).items():
            Litems.append((SL,key))
    t0=_v23c_time.time()
    for ii,(SL,(cs,Esg)) in enumerate(Litems):
        for dv in verts:
            tests+=1
            tcs=_v10a2_sig_canon(_v10a3_translate_sig(cs,dv))
            cand=RB.get((tcs,Esg),())
            if not cand:
                continue
            tSL=_v17_translate_support(SL,dv)
            endpoint=int(_v17_translate_face(int(left_root),tuple(dv)))
            # Endpoint must preserve the left-root plaquette orientation.
            if tuple(faces[endpoint][1:]) != tuple(faces[int(left_root)][1:]):
                raise RuntimeError("support census translation changed endpoint polarization")
            for SR,_ in cand:
                matches+=1
                C=frozenset(set(tSL)|set(SR))
                if int(right_root) not in C:
                    raise RuntimeError("support census lost the anchored ket root")
                if len(C)<=int(max_size) and _v17_connected(C):
                    out.add(C)
                    by_endpoint[endpoint].add(C)
        if V23C_PROGRESS and len(Litems)>200 and (ii+1)%max(1,len(Litems)//5)==0:
            print(f'  support scan {ii+1:,}/{len(Litems):,}; matches={matches:,}; '
                  f'supports={len(out):,}; endpoints={len(by_endpoint):,}; '
                  f'elapsed={_v23c_time.time()-t0:.1f}s')
    return out,{k:set(v) for k,v in by_endpoint.items()},dict(
        blocks=len(Litems),tests=tests,matches=matches,
        supports=len(out),endpoints=len(by_endpoint)
    )


def _v24c_face_translation(f_from,f_to):
    """Lattice translation sending f_from to f_to; orientations must agree."""
    a=faces[int(f_from)]
    b=faces[int(f_to)]
    if tuple(a[1:]) != tuple(b[1:]):
        raise RuntimeError(
            f"cannot translate between different plaquette orientations: {a} -> {b}"
        )
    return tuple((int(b[0][i])-int(a[0][i]))%int(L) for i in range(3))


def _v24c_compose_second_order_supports(seg_by_pair, root_pol):
    """Geometry of the folded {K2,N}/2 term on the diagonal T1 block.

    seg_by_pair[(bra_pol,ket_pol)] is an endpoint-resolved two-W support
    corpus anchored at ket_pol.  For each intermediate polarization b and
    concrete intermediate plaquette y:

        root(a) --two W--> y(b) --two W--> final(a)

    the second segment is translated so its ket anchor b coincides with y.
    This captures cross-polarization intermediate P states and allows the
    true maximal marked support size 7 (= initial + intermediate + final
    P faces + four action faces) instead of the old unsafe <=6 cap.
    """
    a0=int(root_pol)
    out=set()
    composed=0
    size_hist=_v23c_Counter()
    for b in range(3):
        first_map=seg_by_pair[(b,a0)][1]   # endpoint b, ket anchor a0
        second_all=seg_by_pair[(a0,b)][0] # bra a0, ket anchor b
        ket_b=int(anchor_faces[b])
        for y,first_supports in first_map.items():
            dv=_v24c_face_translation(ket_b,int(y))
            second_shifted=[
                _v17_translate_support(S,tuple(dv)) for S in second_all
            ]
            for A in first_supports:
                for Bt in second_shifted:
                    composed+=1
                    U=frozenset(set(A)|set(Bt))
                    if V23C_ROOT not in U:
                        raise RuntimeError("fold support composition lost marked root")
                    if len(U)<=7 and _v17_connected(U):
                        out.add(U)
                        size_hist[len(U)]+=1
    return out,dict(composed_pairs=composed,size_hist=dict(sorted(size_hist.items())))

def _v23c_rooted_connected_subsets(C):
    C=frozenset(map(int,C))
    if V23C_ROOT not in C: return ()
    rest=tuple(sorted(set(C)-{V23C_ROOT})); out=[]
    for mask in range(1<<len(rest)):
        S={V23C_ROOT}
        for i,f in enumerate(rest):
            if (mask>>i)&1: S.add(f)
        F=frozenset(S)
        if _v17_connected(F): out.append(F)
    return tuple(out)

# ------------------------- cubic rooted-shape cache --------------------------
def _v23c_centered_delta(x,x0):
    d=(int(x)-int(x0))%int(L)
    if d>int(L)//2: d-=int(L)//1
    return d

def _v23c_face_rep(f):
    v,a,b=faces[int(f)]; vr=faces[V23C_ROOT][0]
    base=tuple(_v23c_centered_delta(v[i],vr[i]) for i in range(3)); return (base,tuple(sorted((int(a),int(b)))))

def _v23c_face_vertices(rep):
    base,ab=rep; a,b=ab; e1=[0,0,0];e2=[0,0,0];e1[a]=1;e2[b]=1; out=[]
    for s,t in ((0,0),(1,0),(0,1),(1,1)): out.append(tuple(base[i]+s*e1[i]+t*e2[i] for i in range(3)))
    return out

def _v24c_perm_sign(perm):
    inv=sum(int(perm[i])>int(perm[j]) for i in range(3) for j in range(i+1,3))
    return -1 if inv%2 else +1

# Use only proper cubic rotations.  This is sufficient for symmetry reduction
# and avoids importing any parity/pseudovector convention into the cache key.
_V24C_TRANSFORMS=[]
for perm in _v23c_it.permutations(range(3)):
    ps=_v24c_perm_sign(perm)
    for signs in _v23c_it.product((-1,1),repeat=3):
        det=ps*int(signs[0])*int(signs[1])*int(signs[2])
        if det==+1:
            _V24C_TRANSFORMS.append((perm,signs))
if len(_V24C_TRANSFORMS)!=24:
    raise RuntimeError(f"expected 24 proper cubic rotations, got {len(_V24C_TRANSFORMS)}")

def _v23c_transform_vertex(x,T):
    perm,signs=T
    return tuple(int(signs[j])*int(x[perm[j]]) for j in range(3))

def _v23c_transform_face(rep,T):
    vv=[_v23c_transform_vertex(x,T) for x in _v23c_face_vertices(rep)]
    lo=tuple(min(x[i] for x in vv) for i in range(3))
    hi=tuple(max(x[i] for x in vv) for i in range(3))
    axes=tuple(i for i in range(3) if hi[i]-lo[i]==1)
    if len(axes)!=2 or any(hi[i]-lo[i] not in (0,1) for i in range(3)):
        raise RuntimeError('cubic face transform failed')
    return (lo,axes)

def _v23c_shift_rep(rep,d):
    b,ab=rep
    return (tuple(int(b[i])+int(d[i]) for i in range(3)),ab)

def _v24c_shape_key(C):
    """Canonical ROOTED cluster key.

    The old v10a.23 key stored only the transformed unmarked face set.
    If several faces share the root base vertex, it could identify two
    inequivalent choices of marked root.  Here the transformed root face is
    a separate field of the key, so root->root is mandatory.
    """
    C=frozenset(map(int,C))
    if V23C_ROOT not in C:
        raise RuntimeError("rooted shape key called on an unrooted cluster")
    rrep=_v23c_face_rep(V23C_ROOT)
    others=[_v23c_face_rep(f) for f in C if int(f)!=V23C_ROOT]
    keys=[]
    for T in _V24C_TRANSFORMS:
        rr=_v23c_transform_face(rrep,T)
        sh=tuple(-int(x) for x in rr[0])
        rooted=_v23c_shift_rep(rr,sh)
        zz=tuple(sorted(
            _v23c_shift_rep(_v23c_transform_face(r,T),sh)
            for r in others
        ))
        keys.append((rooted,zz))
    return min(keys)


def _v24c_old_unrooted_shape_key(C):
    """Diagnostic only: exact v10a.23 cache key, never used for physics."""
    reps=[_v23c_face_rep(f) for f in C]
    rrep=_v23c_face_rep(V23C_ROOT)
    keys=[]
    # use all 48 transforms to reproduce the old implementation exactly
    for perm in _v23c_it.permutations(range(3)):
        for signs in _v23c_it.product((-1,1),repeat=3):
            T=(perm,signs)
            rr=_v23c_transform_face(rrep,T)
            sh=tuple(-int(x) for x in rr[0])
            zz=tuple(sorted(
                _v23c_shift_rep(_v23c_transform_face(r,T),sh) for r in reps
            ))
            keys.append(zz)
    return min(keys)


def _v24c_root_cache_regression():
    """Geometry-only proof that the v10a.23 unrooted cache was unsafe.

    Enumerate every rooted connected cluster through size four.  Count old
    cache keys that contain more than one genuinely rooted proper-rotation key.
    This test is independent of all Haar/Feshbach numerics.
    """
    levels={1:{frozenset((V23C_ROOT,))}}
    for k in (2,3,4):
        nxt=set()
        for S in levels[k-1]:
            front=set()
            for f in S:
                front.update(_V17_NEIGH[int(f)])
            for g in front-set(S):
                T=frozenset(set(S)|{int(g)})
                if len(T)==k and _v17_connected(T):
                    nxt.add(T)
        levels[k]=nxt
    groups=_v23c_dd(set)
    for C in levels[4]:
        groups[_v24c_old_unrooted_shape_key(C)].add(_v24c_shape_key(C))
    unsafe=sum(len(v)>1 for v in groups.values())
    return unsafe,len(levels[4]),len(groups),len({_v24c_shape_key(C) for C in levels[4]})

# ------------------------------- production ---------------------------------
print('\n[14] INDEPENDENT FOUR-W SUPPORT CENSUS — ROOTED/FULL-T1 CORRECTED')

unsafe_old,n4,old4,new4=_v24c_root_cache_regression()
gate('v10a.24 geometry regression detects the old unrooted shape-cache collision',
     unsafe_old>0,
     f'unsafe old size-4 keys={unsafe_old}; concrete={n4}; old classes={old4}; rooted proper-rotation classes={new4}')

# Direct irreducible fourth-order paths on the marked diagonal polarization.
DMAX,DMAP,DSTATC=_v24c_candidate_supports(
    W2Ls[V23C_POL],W2Ls[V23C_POL],
    anchor_faces[V23C_POL],anchor_faces[V23C_POL],max_size=6
)
CLEFT,CLMAP,CLSTAT=_v24c_candidate_supports(
    W1Ls[V23C_POL],W2Ls[V23C_POL],
    anchor_faces[V23C_POL],anchor_faces[V23C_POL],max_size=6
)
CRIGHT,CRMAP,CRSTAT=_v24c_candidate_supports(
    W2Ls[V23C_POL],W1Ls[V23C_POL],
    anchor_faces[V23C_POL],anchor_faces[V23C_POL],max_size=6
)
CMAX=set(CLEFT)|set(CRIGHT)

# Every two-W model-space segment for all 3x3 T1 polarization pairs.
SEG={}
for b in range(3):
    for a in range(3):
        S,MP,ST=_v24c_candidate_supports(
            W1Ls[b],W1Ls[a],
            anchor_faces[b],anchor_faces[a],max_size=4
        )
        SEG[(b,a)]=(S,MP,ST)
        print(f'  two-W segment bra{b}<-ket{a}: supports={len(S):,}, endpoints={len(MP):,}')

# Same-polarization two-W supports cover J/lower-order geometry.
EMAX=set(SEG[(V23C_POL,V23C_POL)][0])

# Folded normalization support is an OPERATOR CONVOLUTION through every
# intermediate position and polarization, not a union of two root-anchored
# same-polarization sets.
FMAX,FSTAT=_v24c_compose_second_order_supports(SEG,V23C_POL)
print('  folded support composition:',FSTAT)
fold_max=max(map(len,FMAX),default=0)
gate('v10a.24 folded support census allows the full marked O4 support size through 7',
     fold_max<=7,
     f'folded maximal support={fold_max}, supports={len(FMAX):,}')

MAXC=set(DMAX)|set(CMAX)|set(EMAX)|set(FMAX)

# Explicitly include every rooted connected cluster through size 3 so the
# m1/m2/m3 gates do not depend on an O4 H0-match census.
small_levels={1:{frozenset((V23C_ROOT,))}}
for k in (2,3):
    nxt=set()
    for S in small_levels[k-1]:
        front=set()
        for f in S:
            front.update(_V17_NEIGH[int(f)])
        for g in front-set(S):
            T=frozenset(set(S)|{int(g)})
            if len(T)==k and _v17_connected(T):
                nxt.add(T)
    small_levels[k]=nxt
MAXC.update(small_levels[3])

CLUST=set()
for C in MAXC:
    CLUST.update(_v23c_rooted_connected_subsets(C))
CLUST=sorted(CLUST,key=lambda S:(len(S),tuple(sorted(S))))
CLSET=set(CLUST)

badclose=[]
for C in CLUST:
    for S in _v23c_rooted_connected_subsets(C):
        if S not in CLSET:
            badclose.append((C,S))
            break
    if badclose:
        break
gate('v10a.24 independent cluster poset is downward closed',
     not badclose,
     f'clusters={len(CLUST):,}, maximal={len(MAXC):,}')

sizecount=_v23c_Counter(map(len,CLUST))
shapekeys={_v24c_shape_key(C) for C in CLUST}
print('  concrete rooted clusters      =',len(CLUST),dict(sorted(sizecount.items())))
print('  ROOTED proper-rotation classes=',len(shapekeys))
print('  max direct/fold supports      =',
      max(map(len,DMAX),default=0),
      max(map(len,CMAX),default=0),
      max(map(len,FMAX),default=0))

print('\n[15] INDEPENDENT RESTRICTED-HAMILTONIAN COEFFICIENTS')
shape_cache={}; shape_rep={}; duplicate_checks=0; max_fit_stab=0.0; raw={}; t0=_v23c_time.time()
for ci,C in enumerate(CLUST,1):
    key=_v24c_shape_key(C)
    if key not in shape_cache:
        z=_v23c_fit_cluster(C); shape_cache[key]=z; shape_rep[key]=C; max_fit_stab=max(max_fit_stab,z['fit_stability'])
        if V23C_PROGRESS:
            cc=z['coef']; print(f'  shape {len(shape_cache):4d}/{len(shapekeys):4d} |C|={len(C)} dim={z["one_dim"]}/{z["vac_dim"]} '
                  f'q1/q2={z["q1"]}/{z["q2"]} c2={cc[2]:+.9g} c3={cc[3]:+.9g} c4={cc[4]:+.9g} '
                  f'fitΔ4={z["fit_stability"]:.2e} elapsed={_v23c_time.time()-t0:.1f}s',flush=True)
    elif duplicate_checks<V23C_SYM_CHECKS and C!=shape_rep[key]:
        z2=_v23c_fit_cluster(C); z1=shape_cache[key]; err=float(np.max(np.abs(z2['coef'][:5]-z1['coef'][:5])))
        gate(f'v10a.23 cubic-shape duplicate raw coefficients agree #{duplicate_checks+1}',err<2e-5,f'maxerr={err:.3e}'); duplicate_checks+=1
    raw[C]=shape_cache[key]['coef'].copy()
gate('v10a.23 finite-cluster u4 fit window is numerically stable',max_fit_stab<V23C_FIT_STAB_TOL,
     f'max |c4(full)-c4(inner)|={max_fit_stab:.3e}')

print('\n[16] ROOTED INCIDENCE TRANSFORM — INDEPENDENT RAW CLUSTERS')
omega={}; totals=np.zeros(5,float); bysize=_v23c_dd(lambda:np.zeros(5,float))
for C in CLUST:
    x=raw[C].copy()
    for S in _v23c_rooted_connected_subsets(C):
        if S!=C: x-=omega[S]
    omega[C]=x; totals+=x; bysize[len(C)]+=x
for k in sorted(bysize):
    x=bysize[k]; print(f'  size {k}: c1={x[1]:+.12g} c2={x[2]:+.12g} c3={x[3]:+.12g} c4={x[4]:+.12g}')
print('  TOTAL m1/m2/m3/m4 =',totals[1],totals[2],totals[3],totals[4])

gate('v10a.23 independent finite-cluster oracle recovers m1=1',abs(totals[1]-1.0)<2e-5,totals[1])
gate('v10a.23 independent finite-cluster oracle recovers m2=11/306',abs(totals[2]-float(_V23CF(11,306)))<2e-4,totals[2])
gate('v10a.23 independent finite-cluster oracle recovers m3=-109151/249696',abs(totals[3]-float(_V23CF(-109151,249696)))<8e-4,totals[3])

# Every gate from the new v10a.23 layers must pass before disputed values enter memory.
g23=gates[V23_GATE_START:]
if not all(ok for _,ok,_ in g23):
    print('\n'+'='*148); print('V10A.23 STOPPED BEFORE FOURTH-ORDER UNBLIND'); print('='*148)
    for i,(n,ok,d) in enumerate(g23,1): print(f'{i:02d}. {"PASS" if ok else "FAIL"} — {n}'+(f' :: {d}' if d else ''))
    raise AssertionError('v10a.23 pre-unblind gate failure; no fourth-order verdict permitted')

# =============================================================================
# 17. FINAL UNBLIND — disputed constants first appear here
# =============================================================================
print('\n[17] FINAL FOURTH-ORDER UNBLIND')
M4_ORACLE=float(totals[4])
M4_SHORTCUT=_V23CF(-160506019419340168451,14501180577204921600)
Q3_OLD=_V23CF(-20721577909065127111,7250590288602460800)
C3_OLD=_V23CF(-211835444920651,4405310420659200)

dnew=abs(M4_ORACLE-float(M4_SHORTCUT)); dold=abs(M4_ORACLE-float(Q3_OLD))
C_COLD=float(V23_AXIAL_SHAPE['C_direct']); dcold=abs(C_COLD-float(C3_OLD))
print('  independent linked m4         =',repr(M4_ORACLE))
print('  quarantined scalar shortcut   =',M4_SHORTCUT,'=',repr(float(M4_SHORTCUT)),' |Δ|=',dnew)
print('  historical 189-kernel q3      =',Q3_OLD,'=',repr(float(Q3_OLD)),' |Δ|=',dold)
print('  cold folded C_shape           =',repr(C_COLD))
print('  historical C_shape            =',C3_OLD,'=',repr(float(C3_OLD)),' |Δ|=',dcold)

# Convert the independently linked rest scalar into the physical mass kernel by
# a translation-local scalar shift.  Vacuum subtraction cannot change dispersion.
ax_rest=float(V23_AXIAL_SHAPE['rest_direct']); local_shift=M4_ORACLE-ax_rest
K4_mass_cols=V23_AXIAL_H4_COLS.copy()
for a,f in enumerate(anchor_faces): K4_mass_cols[int(f),a]+=local_shift
mass_shape=_v10a3_extract_shape(K4_mass_cols)
record_count=int(np.count_nonzero(np.abs(K4_mass_cols)>V23_RECORD_TOL))
print('  independently linked local shift=',repr(local_shift))
print('  final mass-kernel anchored records=',record_count)
print('  final mass-kernel shape:')
for k in ('rest_direct','A','B','C_direct','D','alpha','fifth_residual_max','hermiticity_error','gamma_spread'):
    print(f'    {k:22s} = {mass_shape[k]:+.15g}')

gate('v10a.23 final physical mass-kernel Gamma rest equals independent cluster oracle',abs(mass_shape['rest_direct']-M4_ORACLE)<2e-6,
     abs(mass_shape['rest_direct']-M4_ORACLE))
gate('v10a.23 final physical kernel has canonical 189 nonzero anchored records',record_count==189,record_count)

if dnew < dold/5.0: scalar_verdict='SCALAR ORACLE SUPPORTS v10a.20 SHORTCUT'
elif dold < dnew/5.0: scalar_verdict='SCALAR ORACLE SUPPORTS HISTORICAL q3'
else: scalar_verdict='SCALAR ORACLE RETURNS THIRD VALUE'
if dcold<2e-4: shape_verdict='FOLDED MATRIX SUPPORTS HISTORICAL C_shape'
else: shape_verdict='FOLDED MATRIX DOES NOT RECOVER HISTORICAL C_shape'

if scalar_verdict.endswith('HISTORICAL q3') and shape_verdict.startswith('FOLDED MATRIX SUPPORTS') and record_count==189:
    VERDICT='DUAL COLD ORACLES SUPPORT THE HISTORICAL 189-RECORD SU(3) O4 KERNEL'
elif scalar_verdict.endswith('v10a.20 SHORTCUT') and not shape_verdict.startswith('FOLDED MATRIX SUPPORTS'):
    VERDICT='DUAL COLD ORACLES SUPPORT THE MODERN SHORTCUT BRANCH'
else:
    VERDICT='MIXED/THIRD RESULT — DO NOT PROMOTE EITHER FOURTH-ORDER CLAIM'

print('\n  SCALAR VERDICT:',scalar_verdict)
print('  SHAPE VERDICT :',shape_verdict)
print('  FINAL VERDICT :',VERDICT)

print('\n'+'='*148)
print('FINAL v10a.23 GATE SUMMARY')
print('='*148)
g23=gates[V23_GATE_START:]
for i,(n,ok,d) in enumerate(g23,1): print(f'{i:02d}. {"PASS" if ok else "FAIL"} — {n}'+(f' :: {d}' if d else ''))
print('-'*148)
print(f'PASSED {sum(ok for _,ok,_ in g23)}/{len(g23)} v10a.23 GATES')
print('finite-cluster shape classes:',len(shape_cache),' concrete clusters:',len(CLUST))
print('independent linked m4       :',repr(M4_ORACLE))
print('folded C_shape              :',repr(C_COLD))
print('189-record count            :',record_count)
print('FINAL VERDICT               :',VERDICT)

if not all(ok for _,ok,_ in g23):
    raise AssertionError('v10a.23 post-unblind structural gate failure; verdict above is diagnostic only')
